<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_09_1_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 9: Foundations of Generative AI**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 9 Material

* **Part 9.1: Transformer Architecture** [[Video]]() [[Notebook]](t81_558_class_09_1_transformers.ipynb)
* Part 9.2: Pretrained Models and the Hugging Face Ecosystem [[Video]]() [[Notebook]](t81_558_class_09_2_hugging.ipynb)
* Part 9.3: Embeddings and Vector Representations [[Video]]() [[Notebook]](t81_558_class_09_3_embedding.ipynb)
* Part 9.4: Diffusion Models and Image Generation [[Video]]() [[Notebook]](t81_558_class_09_4_stable_diff.ipynb)
* Part 9.5: Accessing API-Based LLMs [[Video]]() [[Notebook]](t81_558_class_09_5_chat_gpt.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running the correct version of TensorFlow.
  Running the following code will map your GDrive to ```/content/drive```.

In [1]:
try:
    from google.colab import drive
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: using Google CoLab


# Part 9.1: Introduction to Transformers

Transformers are neural networks that provide state-of-the-art solutions for many of the problems previously assigned to recurrent neural networks. [[Cite:vaswani2017attention]](https://arxiv.org/abs/1706.03762) Sequences can form both the input and the output of a neural network, examples of such configurations include::

* Vector to Sequence - Image captioning
* Sequence to Vector - Sentiment analysis
* Sequence to Sequence - Language translation

Sequence-to-sequence allows an input sequence to produce an output sequence based on an input sequence. Transformers focus primarily on this sequence-to-sequence configuration.

## High-Level Overview of Transformers

This course focuses primarily on the application of deep neural networks. The focus will be on presenting data to a transformer and a transformer's major components. As a result, we will not focus on implementing a transformer at the lowest level. The following section provides an overview of critical internal parts of a transformer, such as residual connections and attention. In the next chapter, we will use transformers from [Hugging Face](https://huggingface.co/) to perform natural language processing with transformers. If you are interested in implementing a transformer from scratch, Keras provides a comprehensive [example](https://www.tensorflow.org/text/tutorials/transformer).

Figure 10.TRANS-1 presents a high-level view of a transformer for language translation.

**Figure 10.TRANS-1: High Level View of a Translation Transformer**
![Transformer](https://data.heatonresearch.com/images/jupyter/transformer-1.jpg)

We use a transformer that translates between English and Spanish for this example. We present the English sentence "the cat likes milk" and receive a Spanish translation of "al gato le gusta la leche."

We begin by placing the English source sentence between the beginning and ending tokens. This input can be of any length, and we presented it to the neural network as a ragged Tensor. Because the Tensor is ragged, no padding is necessary. Such input is acceptable for the attention layer that will receive the source sentence. The encoder transforms this ragged input into a hidden state containing a series of key-value pairs representing the knowledge in the source sentence. The encoder understands to read English and convert to a hidden state. The decoder understands how to output Spanish from this hidden state.

We initially present the decoder with the hidden state and the starting token. The decoder will predict the probabilities of all words in its vocabulary. The word with the highest probability is the first word of the sentence.

The highest probability word is attached concatenated to the translated sentence, initially containing only the beginning token. This process continues, growing the translated sentence in each iteration until the decoder predicts the ending token.

## Transformer Hyperparameters

Before we describe how these layers fit together, we must consider the following transformer hyperparameters, along with default settings from the Keras transformer example:

* num_layers = 4
* d_model = 128
* dff = 512
* num_heads = 8
* dropout_rate = 0.1

Multiple encoder and decoder layers can be present. The **num_layers** hyperparameter specifies how many encoder and decoder layers there are. The expected tensor shape for the input to the encoder layer is the same as the output produced; as a result, you can easily stack these layers.

We will see embedding layers in the next chapter. However, you can think of an embedding layer as a dictionary for now. Each entry in the embedding corresponds to each word in a fixed-size vocabulary. Similar words should have similar vectors. The **d_model** hyperparameter specifies the size of the embedding vector. Though you will sometimes preload embeddings from a project such as [Word2vec](https://radimrehurek.com/gensim/models/word2vec.html) or [GloVe](https://nlp.stanford.edu/projects/glove/), the optimizer can train these embeddings with the rest of the transformer. Training your embeddings allows the **d_model** hyperparameter to set to any desired value. If you transfer the embeddings, you must set the **d_model** hyperparameter to the same value as the transferred embeddings.

The **dff** hyperparameter specifies the size of the dense feedforward layers. The **num_heads** hyperparameter sets the number of attention layers heads. Finally, the dropout_rate specifies a dropout percentage to combat overfitting. We discussed dropout previously in this book.

## Inside a Transformer

In this section, we will examine the internals of a transformer so that you become familiar with essential concepts such as:

* Embeddings
* Positional Encoding
* Attention and Self-Attention
* Residual Connection

You can see a lower-level diagram of a transformer in Figure 10.TRANS-2.

**Figure 10.TRANS-2: Architectural Diagram from the Paper**
![Attention is All you Need](https://data.heatonresearch.com/images/jupyter/transformer-2.jpg)

While the original transformer paper is titled "Attention is All you Need," attention isn't the only layer type you need. The transformer also contains dense layers. However, the title "Attention and Dense Layers are All You Need" isn't as catchy.

The transformer begins by tokenizing the input English sentence. Tokens may or may not be words. Generally, familiar parts of words are tokenized and become building blocks of longer words. This tokenization allows common suffixes and prefixes to be understood independently of their stem word. Each token becomes a numeric index that the transformer uses to look up the vector. There are several special tokens:

* Index 0 = Pad
* Index 1 = Unknow
* Index 2 = Start token
* Index 3 = End token

The transformer uses index 0 when we must pad unused space at the end of a tensor. Index 1 is for unknown words. The starting and ending tokens are provided by indexes 2 and 3.

The token vectors are simply the inputs to the attention layers; there is no implied order or position. The transformer adds the slopes of a sine and cosine wave to the token vectors to encode position.

Attention layers have three inputs: key (k), value(v), and query (q). This layer is self-attention if the query, key, and value are the same. The key and value pairs specify the information that the query operates upon. The attention layer learns what positions of data to focus upon.

The transformer presents the position encoded embedding vectors to the first self-attention segment in the encoder layer. The output from the attention is normalized and ultimately becomes the hidden state after all encoder layers are processed.

The hidden state is only calculated once per query. Once the input Spanish sentence becomes a hidden state, this value is presented repeatedly to the decoder until the decoder forms the final Spanish sentence.

This section presented a high-level introduction to transformers. In the next part, we will implement the encoder and apply it to time series. In the following chapter, we will use [Hugging Face](https://huggingface.co/) transformers to perform natural language processing.

## Translation Example

To make the ideas above concrete, the rest of this notebook builds a small English-to-French translation transformer from scratch in PyTorch. We will train it on a few thousand sentence pairs from the OPUS Books corpus and then ask it to translate a handful of test sentences. The model is intentionally tiny so that it trains in a reasonable amount of time on a single GPU (or even a CPU if you are patient). It will not produce production-quality translations, but it will demonstrate every component we just discussed: token embeddings, positional encoding, an encoder stack, a decoder stack, attention masking for padding, and causal masking for autoregressive decoding.

A few honest expectations before we start. With only a few thousand training pairs, a vocabulary capped at ten thousand tokens, and just a handful of training epochs, the resulting translations will be rough. Common, short sentences may come out reasonably well; anything longer or containing rare vocabulary will quickly degrade. The point of the exercise is not to compete with Google Translate, it is to see every moving part of a transformer in code that fits on a few screens. Once you understand this code, the production-grade transformers from Hugging Face (which we cover in the next part) will feel much less mysterious.


### Step 1: Setup and Loading the Dataset

This first cell handles the housekeeping. We install Hugging Face's `datasets` library, import everything we need (PyTorch, regex tools, and a `Counter` for vocabulary building), and pick a device. If a GPU is available, training will run roughly an order of magnitude faster than on CPU; the code works either way.

We then load `opus_books`, a parallel corpus of English-French sentence pairs drawn from books in the public domain. The first row is printed so you can see the structure: each row is a dictionary with a `translation` field that itself contains `en` and `fr` keys. We are not using a clean, modern translation benchmark on purpose. Book translations are stylistically varied and sometimes quite literary, which makes the task harder for our small model and gives you a realistic sense of where it struggles.


In [2]:
!pip -q install datasets

import math
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from datasets import load_dataset

raw_data = load_dataset(
    "opus_books",
    "en-fr",
    split="train"
)

print(raw_data)
print(raw_data[0])







Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'translation'],
    num_rows: 127085
})
{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}


### Step 2: Subsetting and Filtering the Data

Real translation systems are trained on tens of millions of sentence pairs. That is not realistic for a teaching example, so we cap things aggressively. We keep at most 20,000 pairs, drop any sentence longer than 32 tokens (measured by whitespace splitting), and will later cap the vocabulary at 10,000 tokens per language.

These limits matter for accuracy. Twenty thousand pairs is a tiny dataset by translation standards, and capping length at 32 tokens means the model never sees the long, complex sentences that are common in real text. Both limits keep training fast at the cost of generalization. If you want better results later, the easiest knobs to turn are right here: raise `MAX_EXAMPLES` toward the full dataset (roughly 127,000 pairs) and raise `MAX_LEN` to something like 64 or 128. Expect training time to grow proportionally.


In [3]:
# Use a small subset so this remains a teaching example.

MAX_EXAMPLES = 20000
MAX_LEN = 32
VOCAB_SIZE = 10000

pairs = []

for row in raw_data.select(range(min(MAX_EXAMPLES, len(raw_data)))):
    en = row["translation"]["en"]
    fr = row["translation"]["fr"]

    if len(en.split()) <= MAX_LEN and len(fr.split()) <= MAX_LEN:
        pairs.append((en, fr))

print(f"Usable sentence pairs: {len(pairs):,}")
print(pairs[0])



Usable sentence pairs: 13,483
('The Wanderer', 'Le grand Meaulnes')


### Step 3: Tokenization and Vocabulary Construction

A transformer cannot consume raw text, it needs integer token IDs. This cell defines a deliberately simple word-level tokenizer (lowercase, then split on word characters and punctuation) and builds two vocabularies, one for English and one for French. The four special tokens at the start of each vocabulary are standard: `<pad>` for padding shorter sequences in a batch up to a common length, `<sos>` to mark the start of a sentence, `<eos>` to mark the end, and `<unk>` for any word that did not make it into the top 10,000 most frequent tokens.

This tokenizer is one of the biggest reasons the model's accuracy will be limited. Production transformers use subword tokenizers like Byte Pair Encoding (BPE) or SentencePiece, which break rare words into reusable pieces (so "running" might become "run" + "##ning") and almost never produce `<unk>` tokens. With our word-level scheme, every typo, proper noun, or rare inflection becomes `<unk>` and is effectively invisible to the model. Swapping in a subword tokenizer is one of the highest-impact upgrades you could make to this code.


In [4]:
def tokenize(text):
    text = text.lower().strip()
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)


SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD, SOS, EOS, UNK = range(4)


def build_vocab(texts, max_size):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    most_common = counter.most_common(max_size - len(SPECIALS))
    vocab = SPECIALS + [word for word, count in most_common]

    stoi = {word: i for i, word in enumerate(vocab)}
    itos = {i: word for word, i in stoi.items()}

    return stoi, itos


src_stoi, src_itos = build_vocab([en for en, fr in pairs], VOCAB_SIZE)
tgt_stoi, tgt_itos = build_vocab([fr for en, fr in pairs], VOCAB_SIZE)

print(f"English vocab size: {len(src_stoi):,}")
print(f"French vocab size: {len(tgt_stoi):,}")

English vocab size: 10,000
French vocab size: 10,000


### Step 4: Building the PyTorch Dataset and DataLoaders

This cell wraps our sentence pairs in a PyTorch `Dataset` and `DataLoader`. The `encode` function turns a sentence into a list of integer IDs, sandwiched between the start and end tokens. The custom `collate_batch` function is the interesting bit: because sentences in a batch have different lengths, we use `pad_sequence` to pad them all to the longest length in that batch with the `<pad>` token. The model will later be told to ignore those padding positions when computing attention and loss.

We split the data 90/10 into training and validation. The validation loss tells us whether the model is actually learning something generalizable or just memorizing the training set. With a model this small and a dataset this small, do not be surprised to see validation loss plateau or even rise after a few epochs, which is a classic overfitting signal.


In [5]:
def encode(text, stoi):
    tokens = tokenize(text)
    ids = [SOS]

    for token in tokens:
        ids.append(stoi.get(token, UNK))

    ids.append(EOS)
    return ids


class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, fr = self.pairs[idx]

        src = torch.tensor(encode(en, src_stoi), dtype=torch.long)
        tgt = torch.tensor(encode(fr, tgt_stoi), dtype=torch.long)

        return src, tgt


def collate_batch(batch):
    src_batch, tgt_batch = zip(*batch)

    src_batch = nn.utils.rnn.pad_sequence(
        src_batch, batch_first=True, padding_value=PAD
    )

    tgt_batch = nn.utils.rnn.pad_sequence(
        tgt_batch, batch_first=True, padding_value=PAD
    )

    return src_batch, tgt_batch


train_size = int(0.9 * len(pairs))
train_pairs = pairs[:train_size]
valid_pairs = pairs[train_size:]

train_loader = DataLoader(
    TranslationDataset(train_pairs),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_batch,
)

valid_loader = DataLoader(
    TranslationDataset(valid_pairs),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_batch,
)




### Step 5: Defining the Transformer Model

This is the heart of the example. The `PositionalEncoding` class implements the sine and cosine position encoding from the original paper, the same trick discussed in the section above: because attention has no built-in notion of order, we add a position-dependent vector to each token embedding so the model can tell "the cat sat" apart from "sat the cat".

The `TinyTranslator` class then assembles the full encoder-decoder transformer using PyTorch's built-in `nn.Transformer` module. Notice the hyperparameters: `d_model=128`, `nhead=4`, `num_layers=2`, and `dim_feedforward=512`. Compare these to the defaults from the lecture (`d_model=128`, `num_heads=8`, `num_layers=4`, `dff=512`). We have cut the layer count and head count roughly in half, which gives us a model with on the order of a few million parameters instead of tens of millions. That is the right scale for a 20,000-pair dataset (a much larger model would simply memorize the training data) but it is also a real ceiling on translation quality. Two encoder layers and two decoder layers is not enough capacity to capture the full syntactic structure of either language.

The `forward` method also constructs two important masks. The `key_padding_mask` tells attention to ignore `<pad>` positions, and the `tgt_mask` (a square subsequent mask) prevents the decoder from peeking at future tokens during training, which is what makes the model autoregressive.


In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TinyTranslator(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=512,
        dropout=0.1,
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD)

        self.positional_encoding = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.output_layer = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        src_key_padding_mask = src == PAD
        tgt_key_padding_mask = tgt == PAD

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1),
            device=tgt.device,
        )

        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        return self.output_layer(output)


model = TinyTranslator(
    src_vocab_size=len(src_stoi),
    tgt_vocab_size=len(tgt_stoi),
).to(device)

print(model)


TinyTranslator(
  (src_embedding): Embedding(10000, 128, padding_idx=0)
  (tgt_embedding): Embedding(10000, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=Tru

### Step 6: Training the Model

This cell defines the training and evaluation loops and runs them for five epochs. The key trick to notice is in `train_one_epoch`: we feed the decoder `tgt[:, :-1]` (everything except the last token) as input and ask it to predict `tgt[:, 1:]` (everything except the first token) as output. This is "teacher forcing", a shifted-by-one setup that lets the model learn to predict each next token given the correct previous tokens, and it is what the causal `tgt_mask` from the previous cell enforces.

We use cross-entropy loss with `ignore_index=PAD` so that padding positions contribute nothing to the loss, AdamW as the optimizer with a learning rate of 3e-4 (a sensible default for transformers of this size), and gradient clipping at norm 1.0 to prevent the occasional bad batch from destabilizing training.

What accuracy should you expect? Watch the validation loss. With these settings you will typically see training loss drop steadily while validation loss bottoms out somewhere around epoch 3 to 5 and then starts creeping back up, which is overfitting. Five epochs is a deliberate compromise between "long enough to learn something" and "short enough to wait through in class". Realistic ways to improve this: train for more epochs with a learning rate scheduler (warmup followed by cosine decay is standard for transformers), add label smoothing to the loss, and increase the dataset size before increasing the model size.


In [7]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


def train_one_epoch():
    model.train()
    total_loss = 0

    for src, tgt in train_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        decoder_input = tgt[:, :-1]
        expected_output = tgt[:, 1:]

        logits = model(src, decoder_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_output.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in valid_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            decoder_input = tgt[:, :-1]
            expected_output = tgt[:, 1:]

            logits = model(src, decoder_input)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                expected_output.reshape(-1),
            )

            total_loss += loss.item()

    return total_loss / len(valid_loader)


EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch()
    valid_loss = evaluate()

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f}"
    )

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss: 6.3457 | Valid Loss: 5.3077
Epoch 02 | Train Loss: 5.1339 | Valid Loss: 4.8973
Epoch 03 | Train Loss: 4.7739 | Valid Loss: 4.6918
Epoch 04 | Train Loss: 4.5390 | Valid Loss: 4.5542
Epoch 05 | Train Loss: 4.3651 | Valid Loss: 4.4773


### Step 7: Greedy Decoding for Inference

Training is one thing, but to actually translate a new sentence we need an inference loop. The `translate` function below implements **greedy decoding**: starting from just the `<sos>` token, we repeatedly run the model, take whichever output token has the highest probability, append it to our running output, and feed everything back in. We stop when the model emits `<eos>` or hits the maximum output length.

Greedy decoding is the simplest possible strategy and it is what we use here for clarity, but it is also one of the bigger reasons the translations below will be mediocre. Greedy decoding commits to the locally-best token at every step, even when a slightly worse choice now would set up a much better sentence later. The standard upgrade is **beam search**, which keeps the top `k` partial translations alive at each step (typically `k=4` or `k=5`) and picks the best complete sequence at the end. Beam search alone, with no other changes, often produces noticeably better output. For more recent and creative-feeling output, sampling-based methods like top-k or nucleus (top-p) sampling are also worth knowing about, though they are more common for open-ended generation than for translation.


In [8]:
# Cell #9
def decode(ids, itos):
    words = []

    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, SOS):
            words.append(itos.get(idx, "<unk>"))

    return " ".join(words)


def translate(sentence, max_output_len=40):
    model.eval()

    src = torch.tensor(
        [encode(sentence, src_stoi)],
        dtype=torch.long,
        device=device,
    )

    tgt = torch.tensor([[SOS]], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_output_len):
            logits = model(src, tgt)
            next_token = logits[:, -1, :].argmax(dim=-1).item()

            tgt = torch.cat(
                [
                    tgt,
                    torch.tensor([[next_token]], device=device),
                ],
                dim=1,
            )

            if next_token == EOS:
                break

    return decode(tgt[0].tolist(), tgt_itos)

### Step 8: Trying It Out on Sample Sentences

Here is the payoff. We feed four short, simple English sentences through the trained model and print the French output. We deliberately picked sentences using common, high-frequency vocabulary that should appear many times in the training data: pronouns, the verb "to be", a basic action ("opened the door"), and a basic descriptive sentence ("the house is big").

Be honest about what you see. With this much training data and this small a model, you can expect output that is **grammatically French-shaped** more often than it is **semantically correct**. The model will often produce fluent-looking French that does not actually mean what the English input meant, especially for any input word that ended up as `<unk>`. You may also see repeated tokens, premature `<eos>` predictions, or French sentences with the wrong gender agreement. All of these are characteristic failure modes of small, undertrained transformers and they are exactly the symptoms that the upgrades discussed above (more data, subword tokenization, beam search, more layers, longer training, label smoothing) are designed to fix.

If you want to push this further as an exercise, the suggested order of attack would be: first switch to a subword tokenizer (biggest quality win per line of code), then add beam search to inference, then increase the dataset size, and only then make the model itself bigger. Doing it in that order tends to give you the most improvement for the least compute.


In [9]:
examples = [
    "I am a man.",
    "She is happy.",
    "He opened the door.",
    "The house is big.",
]

for sentence in examples:
    print(f"EN: {sentence}")
    print(f"FR: {translate(sentence)}")
    print()

EN: I am a man.
FR: je suis un peu .

EN: She is happy.
FR: elle est - elle .

EN: He opened the door.
FR: il y avait été .

EN: The house is big.
FR: la porte , la porte .



# Module 9 Assignment

You can find the fifth assignment here: [assignment 9](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class9.ipynb)
